# SF3D base mesh, on a Kaggle GPU

The one 3D lane, run on a T4. Two things this notebook does that the first version did not:

1. **No CLIP download.** The SF3D checkpoint carries the CLIP weights, so `open_clip` only has to build the architecture. The upstream code asks for `laion2b_s34b_b79k` over the hub, which is 577 MB and a way to lose an afternoon.
2. **No token.** The weights come from the ModelScope mirror, which serves the same bytes and needs no gated-repo approval. The DINOv2 image encoder comes from the hub, ungated.

The model is patched to float16 because the T4 has no bf16.

In [ ]:
import os, subprocess, sys, pathlib

REPO = "/kaggle/working/sf3d"
WEIGHTS = "/kaggle/working/weights"
IN_DIR = "/kaggle/input/concepts"
OUT_DIR = "/kaggle/working/out"

subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Stability-AI/stable-fast-3d", REPO], check=True)

# The two C++ extensions need torch at build time: no build isolation.
os.environ["USE_NATIVE_ARCH"] = "0"
os.environ["CPPFLAGS"] = "-include utility"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--no-build-isolation", f"{REPO}/texture_baker", f"{REPO}/uv_unwrapper"], check=True)

# bfloat16 is not supported on the T4.
run_py = pathlib.Path(REPO, "run.py")
run_py.write_text(run_py.read_text().replace("torch.bfloat16", "torch.float16"))

# The CLIP weights are in the checkpoint: build the architecture, download nothing.
clip_py = pathlib.Path(REPO, "sf3d/models/image_estimator/clip_based_estimator.py")
text = clip_py.read_text()
text = text.replace(
    'pretrained = local if local and os.path.exists(local) else self.cfg.pretrain',
    'pretrained = local if local and os.path.exists(local) else None')
clip_py.write_text(text)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "einops==0.7.0", "jaxtyping==0.2.31", "omegaconf==2.3.0", "transformers==4.42.3",
                "open_clip_torch==2.24.0", "trimesh==4.4.1", "numpy==1.26.4", "huggingface-hub==0.23.4",
                "rembg==2.0.57", "gpytoolbox==0.2.0", "pynanoinstantmeshes==0.0.3",
                "gradio==4.41.0", "gradio-litmodel3d==0.0.1"], check=True)
print("deps ready")

In [ ]:
import pathlib, subprocess, os

# The weights from the ModelScope mirror: no token, no gated repo, the same
# bytes the local lane uses. 4024289892 bytes.
os.makedirs(WEIGHTS, exist_ok=True)
base = "https://www.modelscope.cn/models/stabilityai/stable-fast-3d/resolve/master"
for name in ("config.yaml", "model.safetensors"):
    target = pathlib.Path(WEIGHTS, name)
    if target.exists() and target.stat().st_size > 0:
        print("have", name, target.stat().st_size)
        continue
    subprocess.run(["curl", "-L", "--retry", "8", "--retry-delay", "5", "-C", "-",
                    "-o", str(target), f"{base}/{name}"], check=True)
    print("got", name, target.stat().st_size)

cfg = pathlib.Path(WEIGHTS, "config.yaml")
ckpt = pathlib.Path(WEIGHTS, "model.safetensors")
print("weights", cfg.stat().st_size, ckpt.stat().st_size)
assert ckpt.stat().st_size == 4024289892, "weights are not the size they must be"

In [ ]:
import subprocess, sys, os, pathlib, glob
imgs = sorted(glob.glob(f"{IN_DIR}/*.png")) + sorted(glob.glob(f"{IN_DIR}/*.jpg"))
print("images", imgs)
cmd = [sys.executable, "run.py", *imgs, "--output-dir", OUT_DIR,
       "--texture-resolution", "1024", "--foreground-ratio", "0.85",
       "--remesh_option", "none", "--batch_size", "1", "--device", "cuda",
       "--pretrained-model", WEIGHTS]
r = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
print(r.stdout[-3000:])
print(r.stderr[-3000:])
glbs = sorted(glob.glob(f"{OUT_DIR}/**/*.glb", recursive=True))
print("GLBS", glbs)
for g in glbs:
    print("GLSIZE", g, os.path.getsize(g))